In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import sqlite3
import matplotlib.pyplot as plt
from pandas.tseries.offsets import DateOffset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, TunedThresholdClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import fbeta_score, make_scorer
import os

In [ ]:
os.environ['LOKY_MAX_CPU_COUNT'] = "2"
con = sqlite3.connect(database="data.sqlite")


data = pd.read_sql_query(
  sql="SELECT * FROM final_data",
  con=con,
  parse_dates={"Period"}
)
con.close()

In [ ]:
# 100000 distinct loan- and period-specific observations per year
# 17000 distinct loans per 4 years
# 12 period-specific (macro) observations per year
# Ideally, we have enough observations to use a 1-year training window but because we only have 12 macro observations
# we decide to use a 4-year training window instead a smaller window more sensitive to regime changes
# This is the justification for using a 4-year training window, instead of a smaller training window more
# sensitive to regime changes
window = 4
train_start = pd.to_datetime("2012-01-01",format="%Y-%m-%d")

# Our first testing period is 2017, as we keep a 1 year buffer between train and test set due to how our variable is defined
# We will have a monthly rolling window and will compute monthly prediction performance using various statistics
# Gradient boosting contains hyperparameters and so does penalized logit regression. These hyperparameters will be tuned
# once on the first training sample to reduce computation time







In [ ]:
# We will try 3 methods, logit regression, logit regression with penalization, gradient boosting
# Since the latter 2 require hyperparameters, we do a hyperparameter search on the initial training set using cross validation

train = data.loc[(data["Period"] >= train_start) & (data["Period"] < train_start + DateOffset(years=window)),
                 ~data.columns.isin(["Loan Identifier","Postal Code","Period"])]


# Penalties require data to be on same scale, and standardization does not affect logit or tree-based methods
def standardize_data(X_train,X_test):
    X_tr = X_train.copy()
    X_te = X_test.copy()

    scaler = StandardScaler()
    is_numeric = X_tr.apply(lambda x: not ((x==0) | (x==1)).all(), axis=0)
    numeric_cols = X_tr.columns[is_numeric]
    X_tr[numeric_cols] = X_tr.loc[:,numeric_cols].astype(float)
    X_te[numeric_cols] = X_te.loc[:,numeric_cols].astype(float)

    scaler.fit(X_tr.loc[:,numeric_cols])
    X_tr.loc[:,numeric_cols] = scaler.transform(X_tr.loc[:,numeric_cols])
    X_te.loc[:,numeric_cols] = scaler.transform(X_te.loc[:,numeric_cols])

    return X_tr, X_te, scaler.mean_, scaler.scale_

def hyperparameter_tuning(X_train,y_train,estimator,param_grid,folds):

    # Both false negatives (missing loan default) and than false positives (requiring more reserves than necessary,
    # or assisting loan repayment with those who do not need it) are important so we use a both precision- and recall-based measure
    gs = GridSearchCV(estimator,param_grid,cv=folds,scoring="average_precision").fit(X_train,y_train)
    return gs.best_params_

# We also want to decide upon a probability threshold for defaults
def decision_threshold_tuning(X_train,y_train,estimator,folds):

    # Here, we care about the fact that recall is more important than precision, so we use a relevant measure
    f2_scorer = make_scorer(fbeta_score, beta=2)
    tcv = TunedThresholdClassifierCV(estimator,cv=folds,scoring=f2_scorer).fit(X_train,y_train)

    return tcv.best_threshold_
    




In [ ]:

X_train = train.loc[:,train.columns!="Default Next 12"]
y_train = train.loc[:,"Default Next 12"]

X_train_s, _, _, _ = standardize_data(X_train,X_train)

# C=0.001
est_lp = LogisticRegression(l1_ratio=0)
params = {"C":[0.001,0.01,0.1,1,10,100,1000]}
print(f"Best hyperparameters for penalized logit: {hyperparameter_tuning(X_train_s,y_train,est_lp,params,4)}")

# learning_rate=0.03,max_depth=None,max_iter=300
est_gb = HistGradientBoostingClassifier(random_state=0)
params = {"max_iter":[100,200,300],"learning_rate":[0.03,0.05,0.1],"max_depth":[None,10]}
print(f"Best hyperparameters for gradient boosting: {hyperparameter_tuning(X_train_s,y_train,est_gb,params,4)}")

In [ ]:
# Now we know hyperparameter values, we tune decision threshold with those hyperparameters when relevant
# Standard logit threshold of 0.010
est_l = LogisticRegression(C=np.inf)
print(f"Best decision threshold for logit: {decision_threshold_tuning(X_train_s,y_train,est_l,4)}")
# Penalized logit threshold of 0.010
est_lp = LogisticRegression(l1_ratio = 0, C = 100)
print(f"Best decision threshold for penalized logit: {decision_threshold_tuning(X_train_s,y_train,est_lp,4)}")
# Gradient boosting threshold of 0.010
est_gb = HistGradientBoostingClassifier(random_state=0,max_iter=200,learning_rate=0.05,max_depth=None)
print(f"Best decision threshold for gradient boosting: {decision_threshold_tuning(X_train_s,y_train,est_gb,4)}")



In [ ]:
# Need to replace cross-validation with regular validation like a pre-training set data (2007-2010 then 2011 validation)
